In [2]:
import os
import glob
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

source of files: https://portalsivigila.ins.gov.co/Paginas/Buscador.aspx

# Dengue

In [3]:
# read files
folder_path = "Dengue"  # Adjust this to your folder path

files_dengue = [file_path for file_path in glob.glob(os.path.join(folder_path, "*")) if os.path.isfile(file_path)]

In [4]:
# Assuming files_dengue is a list of file paths to Excel files
dataframes = []

cols_to_keep = ['SEMANA', 'ANO','Estado_final_de_caso','Nom_upgd' ,'Pais_ocurrencia', 'Departamento_ocurrencia', 'Municipio_ocurrencia']

for file in tqdm(files_dengue):
    try:
        df_temp = pd.read_excel(file)
        df_temp = df_temp[cols_to_keep]
        df_temp = df_temp[df_temp['Estado_final_de_caso']!=2]
        # create index column for clinics
        df_temp['idx'] = df_temp['Nom_upgd'].astype(str) + '_' + df_temp['Departamento_ocurrencia'].astype(str) + '_' + df_temp['Municipio_ocurrencia'].astype(str)
        # groupby
        df_group = df_temp.groupby(['SEMANA', 'ANO', 'idx']).size().reset_index(name='count').reset_index(drop=True)
        
        dataframes.append(df_group)
    except Exception as e:
        print(f"Error reading {file}: {e}")

100%|██████████| 17/17 [16:47<00:00, 59.26s/it]


In [5]:
# Concatenate all DataFrames into a single DataFrame
df = pd.concat(dataframes, ignore_index=True)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351249 entries, 0 to 351248
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   SEMANA  351249 non-null  int64 
 1   ANO     351249 non-null  int64 
 2   idx     351249 non-null  object
 3   count   351249 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 10.7+ MB


In [7]:
df.head()

,SEMANA,ANO,idx,count
0,1,2007,AD SALUD LTDA_VALLE_PALMIRA,2
1,1,2007,ALCALDIA MUNICIPAL CAMPOALEGRE_HUILA_CAMPOALEGRE,1
2,1,2007,CAJA DE COMPENSACION FAMILIAR DEL VALLE COMFAM...,3
3,1,2007,CENTRO DE ATENCION EL CASTILLO_META_EL CASTILLO,1
4,1,2007,CENTRO DE SALUD OLAYA HERRERA_MAGDALENA_* MAGD...,1


In [8]:
# Create a Date column (setting the first day of the week)
df["DATE"] = pd.to_datetime(df["ANO"].astype(str) + "-W" + df["SEMANA"].astype(str) + "-1", format="%G-W%V-%u")
df.sort_values(by='DATE', inplace = True)
df.head()

,SEMANA,ANO,idx,count,DATE
0,1,2007,AD SALUD LTDA_VALLE_PALMIRA,2,2007-01-01
106,1,2007,ESE MARIA AUXILIADORA DE GARZON_HUILA_GARZON,1,2007-01-01
107,1,2007,ESE SALUD YOPAL_CASANARE_YOPAL,2,2007-01-01
108,1,2007,ESE SANTA ROSA DE LIMA DE PAICOL_HUILA_PAICOL,1,2007-01-01
109,1,2007,ESE SOR TERESA ADELE SEDE DONCELLO_CALDAS_* CA...,1,2007-01-01


In [9]:
df.to_csv('silver/dengue_spatial.csv', index=False, encoding='utf-8-sig')

# ZIKA

In [10]:
# read files
folder_path = "Zika"  # Adjust this to your folder path

files_dengue = [file_path for file_path in glob.glob(os.path.join(folder_path, "*")) if os.path.isfile(file_path)]

In [11]:
# Assuming files_dengue is a list of file paths to Excel files
dataframes = []

cols_to_keep = ['SEMANA', 'ANO','Estado_final_de_caso','Nom_upgd' ,'Pais_ocurrencia', 'Departamento_ocurrencia', 'Municipio_ocurrencia']

for file in tqdm(files_dengue):
    try:
        df_temp = pd.read_excel(file)
        df_temp = df_temp[cols_to_keep]
        df_temp = df_temp[(df_temp['Estado_final_de_caso']!=2) & (df_temp['Estado_final_de_caso']!=1)]
        # create index column for clinics
        df_temp['idx'] = df_temp['Nom_upgd'].astype(str) + '_' + df_temp['Departamento_ocurrencia'].astype(str) + '_' + df_temp['Municipio_ocurrencia'].astype(str)
        # groupby
        df_group = df_temp.groupby(['SEMANA', 'ANO', 'idx']).size().reset_index(name='count').reset_index(drop=True)
        
        dataframes.append(df_group)
    except Exception as e:
        print(f"Error reading {file}: {e}")

100%|██████████| 9/9 [01:32<00:00, 10.28s/it]


In [12]:
# Concatenate all DataFrames into a single DataFrame
df = pd.concat(dataframes, ignore_index=True)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23319 entries, 0 to 23318
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   SEMANA  23319 non-null  int64 
 1   ANO     23319 non-null  int64 
 2   idx     23319 non-null  object
 3   count   23319 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 728.8+ KB


In [14]:
df.head()

,SEMANA,ANO,idx,count
0,32,2015,COMFANDI IPS PASOANCHO_VALLE_CALI,1
1,32,2015,CORPORACION IPS SALUDCOOP_NORTE SANTANDER_CUCUTA,1
2,32,2015,ESE HOSPITAL JUAN LUIS LONDOÑOG_NORTE SANTANDE...,1
3,32,2015,ESE IMSALUD_NORTE SANTANDER_CUCUTA,11
4,32,2015,SECRETARIA DE SALUD DEPARTAMENTAL_SAN ANDRES_S...,1


In [15]:
# Create a Date column (setting the first day of the week)
df["DATE"] = pd.to_datetime(df["ANO"].astype(str) + "-W" + df["SEMANA"].astype(str) + "-1", format="%G-W%V-%u")
df.sort_values(by='DATE', inplace = True)
df.head()

,SEMANA,ANO,idx,count,DATE
0,32,2015,COMFANDI IPS PASOANCHO_VALLE_CALI,1,2015-08-03
1,32,2015,CORPORACION IPS SALUDCOOP_NORTE SANTANDER_CUCUTA,1,2015-08-03
2,32,2015,ESE HOSPITAL JUAN LUIS LONDOÑOG_NORTE SANTANDE...,1,2015-08-03
3,32,2015,ESE IMSALUD_NORTE SANTANDER_CUCUTA,11,2015-08-03
4,32,2015,SECRETARIA DE SALUD DEPARTAMENTAL_SAN ANDRES_S...,1,2015-08-03


In [16]:
df.to_csv('silver/zika_spatial.csv', index=False, encoding='utf-8-sig')

# Chicunguya

In [17]:
# read files
folder_path = "Chicunguya"  # Adjust this to your folder path

files_dengue = [file_path for file_path in glob.glob(os.path.join(folder_path, "*")) if os.path.isfile(file_path)]

In [18]:
# Assuming files_dengue is a list of file paths to Excel files
dataframes = []

cols_to_keep = ['SEMANA', 'ANO','Estado_final_de_caso','Nom_upgd' ,'Pais_ocurrencia', 'Departamento_ocurrencia', 'Municipio_ocurrencia']

for file in tqdm(files_dengue):
    try:
        df_temp = pd.read_excel(file)
        df_temp = df_temp[cols_to_keep]
        df_temp = df_temp[(df_temp['Estado_final_de_caso']!=2) & (df_temp['Estado_final_de_caso']!=1)]
        # create index column for clinics
        df_temp['idx'] = df_temp['Nom_upgd'].astype(str) + '_' + df_temp['Departamento_ocurrencia'].astype(str) + '_' + df_temp['Municipio_ocurrencia'].astype(str)
        # groupby
        df_group = df_temp.groupby(['SEMANA', 'ANO', 'idx']).size().reset_index(name='count').reset_index(drop=True)
        
        dataframes.append(df_group)
    except Exception as e:
        print(f"Error reading {file}: {e}")

100%|██████████| 10/10 [00:12<00:00,  1.30s/it]


In [19]:
# Concatenate all DataFrames into a single DataFrame
df = pd.concat(dataframes, ignore_index=True)

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28914 entries, 0 to 28913
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   SEMANA  28914 non-null  int64 
 1   ANO     28914 non-null  int64 
 2   idx     28914 non-null  object
 3   count   28914 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 903.7+ KB


In [21]:
df.head()

,SEMANA,ANO,idx,count
0,23,2014,CLINICA SAN FRANCISCO SA_VALLE_TULUA,1
1,23,2014,IPS UNIVERSITARIA HOSPITAL GENERAL DE BARRANQU...,2
2,23,2014,IPS UNIVERSITARIA SEDE CAMINO SALUD METROPOLIT...,1
3,23,2014,IPS UNIVERSITARIA SEDE CAMINO SIMON BOLIVAR_AT...,8
4,23,2014,IPS UNIVERSITARIA SEDE PASO LAS FLORES_ATLANTI...,1


In [22]:
# Create a Date column (setting the first day of the week)
df["DATE"] = pd.to_datetime(df["ANO"].astype(str) + "-W" + df["SEMANA"].astype(str) + "-1", format="%G-W%V-%u")
df.sort_values(by='DATE', inplace = True)
df.head()

,SEMANA,ANO,idx,count,DATE
0,23,2014,CLINICA SAN FRANCISCO SA_VALLE_TULUA,1,2014-06-02
1,23,2014,IPS UNIVERSITARIA HOSPITAL GENERAL DE BARRANQU...,2,2014-06-02
2,23,2014,IPS UNIVERSITARIA SEDE CAMINO SALUD METROPOLIT...,1,2014-06-02
3,23,2014,IPS UNIVERSITARIA SEDE CAMINO SIMON BOLIVAR_AT...,8,2014-06-02
4,23,2014,IPS UNIVERSITARIA SEDE PASO LAS FLORES_ATLANTI...,1,2014-06-02


In [23]:
df.to_csv('silver/chikungunya_spatial.csv', index=False, encoding='utf-8-sig')

# Varicela

In [24]:
# read files
folder_path = "Varicela"  # Adjust this to your folder path

files_dengue = [file_path for file_path in glob.glob(os.path.join(folder_path, "*")) if os.path.isfile(file_path)]

In [25]:
# Assuming files_dengue is a list of file paths to Excel files
dataframes = []

cols_to_keep = ['SEMANA', 'ANO','Estado_final_de_caso','Nom_upgd' ,'Pais_ocurrencia', 'Departamento_ocurrencia', 'Municipio_ocurrencia']

for file in tqdm(files_dengue):
    try:
        df_temp = pd.read_excel(file)
        df_temp = df_temp[cols_to_keep]
        df_temp = df_temp[(df_temp['Estado_final_de_caso']!=2) & (df_temp['Estado_final_de_caso']!=1)]
        # create index column for clinics
        df_temp['idx'] = df_temp['Nom_upgd'].astype(str) + '_' + df_temp['Departamento_ocurrencia'].astype(str) + '_' + df_temp['Municipio_ocurrencia'].astype(str)
        # groupby
        df_group = df_temp.groupby(['SEMANA', 'ANO', 'idx']).size().reset_index(name='count').reset_index(drop=True)
        
        dataframes.append(df_group)
    except Exception as e:
        print(f"Error reading {file}: {e}")

100%|██████████| 17/17 [18:10<00:00, 64.13s/it] 


In [26]:
# Concatenate all DataFrames into a single DataFrame
df = pd.concat(dataframes, ignore_index=True)

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 589111 entries, 0 to 589110
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   SEMANA  589111 non-null  int64 
 1   ANO     589111 non-null  int64 
 2   idx     589111 non-null  object
 3   count   589111 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 18.0+ MB


In [28]:
df.head()

,SEMANA,ANO,idx,count
0,1,2007,AG SERVICIOS DE SALUD LTDA_BOGOTA_BOGOTA,1
1,1,2007,ASISTIR SALUD LTDA FONTIBON_BOGOTA_BOGOTA,1
2,1,2007,BOGOTA Ambito Familiar USAQUEN_BOGOTA_BOGOTA,1
3,1,2007,CAFI COOMULTRASAN_SANTANDER_GIRON,1
4,1,2007,CAJA COMPENSACION FAMILIAR CAFAM FLORESTA_BOGO...,2


In [29]:
# Create a Date column (setting the first day of the week)
df["DATE"] = pd.to_datetime(df["ANO"].astype(str) + "-W" + df["SEMANA"].astype(str) + "-1", format="%G-W%V-%u")
df.sort_values(by='DATE', inplace = True)
df.head()

,SEMANA,ANO,idx,count,DATE
0,1,2007,AG SERVICIOS DE SALUD LTDA_BOGOTA_BOGOTA,1,2007-01-01
74,1,2007,HOSPITAL PABLO VI BOSA ESE UBA SAN JOAQUIN_BOG...,1,2007-01-01
73,1,2007,HOSPITAL PABLO VI BOSA ESE UBA SAN BERNARDINO_...,1,2007-01-01
72,1,2007,HOSPITAL PABLO VI BOSA ESE UBA EL TOCHE_BOGOTA...,3,2007-01-01
71,1,2007,HOSPITAL PABLO VI BOSA ESE - CAMI_BOGOTA_BOGOTA,3,2007-01-01


In [30]:
df.to_csv('silver/varicela_spatial.csv', index=False, encoding='utf-8-sig')